# Notebook 09: Advanced JAX Transforms

## Why This Matters

Beyond `grad`, `vmap`, and `jit`, JAX provides advanced differentiation primitives that unlock entirely new classes of algorithms. The Jacobian and Hessian enable second-order optimization. `custom_vjp` lets you override the backward pass for numerical stability (essential for log-sum-exp, softmax, etc.). Gradient checkpointing (`jax.checkpoint`) enables training larger models than would otherwise fit in memory. These tools are what distinguish JAX power users from casual users.


In [ ]:
%pip install -q jax jaxlib numpy matplotlib

In [ ]:
import jax
import jax.numpy as jnp
from jax import lax
import numpy as np
import matplotlib.pyplot as plt
import time
print("Ready.")

## 1. jax.jacobian: Full Jacobian Matrix

The Jacobian $J_{ij} = \frac{\partial f_i}{\partial x_j}$ is the generalization of the gradient to vector-valued functions. It has shape (output_dim, input_dim).

Two modes:
- `jax.jacfwd`: forward-mode AD -- efficient when output_dim >> input_dim
- `jax.jacrev`: reverse-mode AD -- efficient when output_dim << input_dim (standard case)

For square Jacobians, either works. For tall Jacobians (more outputs than inputs), use forward mode.


In [ ]:
# Jacobian computation
def f(x):
    return jnp.array([
        x[0] ** 2 + x[1],
        x[0] * x[1],
        jnp.sin(x[0]) + x[1] ** 2
    ])

x = jnp.array([1.0, 2.0])

# Reverse-mode Jacobian (efficient for n_out < n_in)
J_rev = jax.jacrev(f)(x)
# Forward-mode Jacobian (efficient for n_in < n_out)
J_fwd = jax.jacfwd(f)(x)

print('Jacobian (reverse mode):')
print(J_rev)
print('\nJacobian (forward mode):')
print(J_fwd)
print('\nAre they equal:', jnp.allclose(J_rev, J_fwd))

# Manual verification
# J[0,0] = d(x0^2 + x1)/dx0 = 2*x0 = 2.0
# J[1,0] = d(x0*x1)/dx0 = x1 = 2.0
# J[2,0] = d(sin(x0) + x1^2)/dx0 = cos(x0) = cos(1)
print(f'\nManual check:')
print(f'J[0,0] = 2*x0 = {2*x[0]:.4f}, computed: {J_rev[0,0]:.4f}')
print(f'J[1,0] = x1   = {x[1]:.4f}, computed: {J_rev[1,0]:.4f}')
print(f'J[2,0] = cos(x0) = {jnp.cos(x[0]):.4f}, computed: {J_rev[2,0]:.4f}')

## 2. Neural Tangent Kernel (NTK): Connecting Infinite Networks to Kernels

The NTK is a profound theoretical result (Jacot et al. 2018): infinitely wide neural networks are equivalent to kernel machines, with a specific kernel called the Neural Tangent Kernel.

$$K(x, x') = J(x) \cdot J(x')^T$$

where $J(x)$ is the Jacobian of the network outputs with respect to all parameters.

Computing the NTK requires computing Jacobians of the network w.r.t. parameters -- a naturally JAX operation. It can be computed as:
```python
J = jax.vmap(jax.jacobian(f_wrt_params))(X)
NTK = J @ J.T  # (N, N) kernel matrix
```


In [ ]:
# Computing the Neural Tangent Kernel

def mlp_fn(params, x):
    W1, b1, W2, b2 = params
    h = jnp.tanh(x @ W1 + b1)
    return h @ W2 + b2

# Initialize params
key = jax.random.PRNGKey(0)
k1, k2 = jax.random.split(key)
W1 = jax.random.normal(k1, (4, 8)) / jnp.sqrt(4)
b1 = jnp.zeros(8)
W2 = jax.random.normal(k2, (8, 1)) / jnp.sqrt(8)
b2 = jnp.zeros(1)
params = (W1, b1, W2, b2)

# Jacobian of output w.r.t. params (flattened)
def get_jac(x):
    jac = jax.jacobian(lambda p: mlp_fn(p, x))(params)
    # Flatten all parameter gradients into one vector
    flat_jac = jnp.concatenate([j.ravel() for j in jax.tree_util.tree_leaves(jac)])
    return flat_jac  # shape: (output_dim * n_params,)

# Compute NTK for a small dataset
X = jax.random.normal(key, (8, 4))  # 8 examples, 4 features

# Jacobian per example
J = jax.vmap(get_jac)(X)  # (n_examples, n_params)
print('Jacobian matrix shape:', J.shape)

# NTK = J @ J^T
NTK = J @ J.T  # (n_examples, n_examples)
print('NTK shape:', NTK.shape)

# NTK should be symmetric positive semi-definite
eigenvalues = jnp.linalg.eigvalsh(NTK)
print('NTK eigenvalues:', eigenvalues)
print('PSD (all >= 0):', jnp.all(eigenvalues >= -1e-6))

## 3. custom_vjp: Overriding the Backward Pass

Sometimes the default autograd backward pass is numerically unstable. For example, the gradient of `log(sum(exp(x)))` involves cancellation. The standard trick is log-sum-exp, but implementing it in a way that JAX's autograd also uses the stable form requires `custom_vjp`.

`custom_vjp` lets you define a **custom backward pass** for any function. This is critical for:
- Numerically stable operations (log-softmax, sigmoid with extreme inputs)
- Implicit differentiation through iterative solvers
- Gradient checkpointing at the function level


In [ ]:
# custom_vjp: numerically stable log-softmax

@jax.custom_vjp
def stable_logsumexp(x):
    c = jnp.max(x)
    return c + jnp.log(jnp.sum(jnp.exp(x - c)))

def stable_logsumexp_fwd(x):
    result = stable_logsumexp(x)
    return result, (x, result)  # (output, residuals for backward)

def stable_logsumexp_bwd(residuals, g):
    x, result = residuals
    # Gradient of log(sum(exp(x))): softmax(x)
    grad = jnp.exp(x - result)
    return (g * grad,)  # tuple matching number of primal inputs

stable_logsumexp.defvjp(stable_logsumexp_fwd, stable_logsumexp_bwd)

# Test with extreme values
x = jnp.array([1000.0, 1001.0, 1002.0])

# Standard log(sum(exp(x))): overflows with large values
try:
    naive = jnp.log(jnp.sum(jnp.exp(x)))
    print(f'Naive logsumexp: {naive} (inf = overflow)')
except:
    print('Naive logsumexp: overflow')

stable = stable_logsumexp(x)
print(f'Stable logsumexp: {stable}')
expected = 1002.0 + jnp.log(jnp.exp(-2) + jnp.exp(-1) + 1)
print(f'Expected: {expected}')

# Gradient of stable version
grad_stable = jax.grad(stable_logsumexp)(x)
print(f'Gradient (should be softmax): {grad_stable}')
print(f'jax.nn.softmax: {jax.nn.softmax(x)}')
print(f'Match: {jnp.allclose(grad_stable, jax.nn.softmax(x))}')

## 4. jax.checkpoint: Gradient Checkpointing

The forward pass stores activations for use in the backward pass. For deep networks, this can use all available memory. Gradient checkpointing (rematerialization) trades compute for memory: recompute activations during the backward pass instead of storing them.

In JAX, `jax.checkpoint` (also called `jax.remat`) marks a function for rematerialization. The rule: wrap functions that produce large intermediate tensors but are cheap to recompute (like attention layers).

Memory savings: O(sqrt(N)) for a depth-N network with checkpointing at sqrt(N) points.


In [ ]:
# jax.checkpoint: trade compute for memory

def attention_layer(x, W_q, W_k, W_v):
    # Simplified self-attention
    Q = x @ W_q
    K = x @ W_k
    V = x @ W_v
    scores = Q @ K.T / jnp.sqrt(K.shape[-1])
    attn = jax.nn.softmax(scores, axis=-1)
    return attn @ V

# Checkpointed version: recompute attention activations during backward
@jax.checkpoint
def attention_layer_checkpointed(x, W_q, W_k, W_v):
    Q = x @ W_q
    K = x @ W_k
    V = x @ W_v
    scores = Q @ K.T / jnp.sqrt(K.shape[-1])
    attn = jax.nn.softmax(scores, axis=-1)
    return attn @ V

key = jax.random.PRNGKey(0)
seq_len, d_model = 64, 32
x = jax.random.normal(key, (seq_len, d_model))
W_q = jax.random.normal(key, (d_model, d_model)) * 0.1
W_k = jax.random.normal(key, (d_model, d_model)) * 0.1
W_v = jax.random.normal(key, (d_model, d_model)) * 0.1

# Verify both produce same output
out1 = attention_layer(x, W_q, W_k, W_v)
out2 = attention_layer_checkpointed(x, W_q, W_k, W_v)
print('Outputs match:', jnp.allclose(out1, out2, atol=1e-5))

# Both produce same gradients
def loss_fn(W_q, W_k, W_v):
    return jnp.sum(attention_layer(x, W_q, W_k, W_v))

def loss_fn_ckpt(W_q, W_k, W_v):
    return jnp.sum(attention_layer_checkpointed(x, W_q, W_k, W_v))

grad1 = jax.grad(loss_fn)(W_q, W_k, W_v)
grad2 = jax.grad(loss_fn_ckpt)(W_q, W_k, W_v)
print('Gradients match:', jnp.allclose(grad1[0], grad2[0], atol=1e-5))
print('(checkpoint recomputes forward, but gives identical gradients)')

## 5. Meta-Learning with JAX: MAML Inner Loop

MAML (Model-Agnostic Meta-Learning) requires computing gradients of gradients -- the outer loop differentiates through the inner loop optimization. In PyTorch this requires `higher` library. In JAX, it is natural: `grad(grad(loss))`.


In [ ]:
# MAML inner loop: grad through grad

def loss_fn(params, x, y):
    W, b = params
    pred = x @ W + b
    return jnp.mean((pred - y) ** 2)

def inner_update(params, x_support, y_support, lr=0.1, n_steps=5):
    """MAML inner loop: adapt params on support set."""
    for _ in range(n_steps):
        grads = jax.grad(loss_fn)(params, x_support, y_support)
        params = jax.tree_util.tree_map(lambda p, g: p - lr * g, params, grads)
    return params

def maml_loss(meta_params, support_x, support_y, query_x, query_y):
    """Outer loss: evaluate adapted params on query set."""
    adapted_params = inner_update(meta_params, support_x, support_y)
    return loss_fn(adapted_params, query_x, query_y)

# Meta-gradient: gradient of the outer loss w.r.t. initial params
meta_grad_fn = jax.jit(jax.grad(maml_loss))

key = jax.random.PRNGKey(0)
k1, k2, k3 = jax.random.split(key, 3)

meta_params = (jax.random.normal(k1, (4, 1)), jnp.zeros(1))
support_x = jax.random.normal(k2, (8, 4))
support_y = jax.random.normal(k2, (8, 1))
query_x   = jax.random.normal(k3, (8, 4))
query_y   = jax.random.normal(k3, (8, 1))

meta_grads = meta_grad_fn(meta_params, support_x, support_y, query_x, query_y)
print('MAML meta-gradient W shape:', meta_grads[0].shape)
print('MAML meta-gradient b shape:', meta_grads[1].shape)
print('Meta-grad W norm:', jnp.linalg.norm(meta_grads[0]))
print('\nThis gradient flows THROUGH the inner optimization loop.')
print('In PyTorch, this requires torch.func or the higher library.')
print('In JAX, it is just grad applied to a function that itself calls grad.')

## Summary

### Key Concepts

| Transform | Use case | Key API |
|-----------|----------|---------|
| `jax.jacobian` | Vector-valued functions | `jacrev` (backward) or `jacfwd` (forward) |
| Neural Tangent Kernel | Theory, kernel methods | `vmap(jacobian(f))` |
| `custom_vjp` | Stable backward pass | `.defvjp(fwd_fn, bwd_fn)` |
| `jax.checkpoint` | Memory-compute tradeoff | Wrap expensive-to-store layers |
| MAML | Meta-learning | `grad` applied to a function that calls `grad` |

**Key insight**: JAX's composability extends to meta-learning -- differentiating through optimization loops is just function composition. No special library needed.

**Next**: `10_capstone_transformer_in_jax.ipynb` -- building a full transformer in JAX.
